In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark import SparkContext, SparkConf
import re

def normalize_words(text):
    return re.compile(r'\W+',re.UNICODE).split(text.lower())

In [3]:
spark_conf = (SparkConf()
    .setMaster('local[1]')
    .setAppName('20')
    .set('spark.driver.host', '127.0.0.1')
    .set('spark.driver.bindAddress', '127.0.0.1')
    .set('spark.pyspark.python', sys.executable)
    .set('spark.python.worker.reuse', 'false')
    .set('spark.ui.enabled', 'false'))

In [4]:
spark = SparkContext.getOrCreate(conf=spark_conf)

In [5]:
data_path = r'C:\datas\3rd_week_data\1800.csv'
lines = spark.textFile(data_path)

In [6]:
lines.first()

'ITE00100554,18000101,TMAX,-75,,,E,'

- **Station ID, 날짜, 온도 타입, 온도(화씨), 기타1, 기타2, 기타3**
- **Station ID, 온도 타입, 온도**를 추출하되, 온도는 섭씨로 변환하여 추출함
- 섭씨온도 = 화씨온도 * 9 / 5 + 32

In [7]:
def parseLine(line):
    # csv 파일은 콤마로 분리되어 있음
    fields = line.split(',')
    # fields[0]: ID, fields[1]: 날짜, fields[2]: 온도 타입, fields[3]: 온도(화씨)
    # 화씨온도(fields의 네번째 값)를 섭씨로 변경
    c_temp = float(fields[3]) * 0.1 * 9 / 5 + 32 # 0.1은 데이터가 전부 10이 곱해진 값이라 해주는 거
    return fields[0], (fields[2], c_temp)
    # fields[0](ID)는 key, (fields[2](온도타입), c_temp(섭씨온도))는 value로 반환

In [8]:
map_line = lines.map(parseLine)

In [9]:
map_line.take(5)

[('ITE00100554', ('TMAX', 18.5)),
 ('ITE00100554', ('TMIN', 5.359999999999996)),
 ('GM000010962', ('PRCP', 32.0)),
 ('EZE00100082', ('TMAX', 16.520000000000003)),
 ('EZE00100082', ('TMIN', 7.699999999999999))]

## (1) 온도타입이 **TMIN**인 데이터만 추출
- `RDD.filter(func)`: ㅎ마수의 결과가 True인 line만 추출
> - `map_line[1]`: **value**(온도타입, 섭씨온도), `map_line[1][0]`: **온도타입**
> - `map_line[0]`: **key**(StationID)

In [10]:
min_temp = map_line.filter(lambda x:x[1][0]=='TMIN')
min_temp.take(5)

[('ITE00100554', ('TMIN', 5.359999999999996)),
 ('EZE00100082', ('TMIN', 7.699999999999999)),
 ('ITE00100554', ('TMIN', 9.5)),
 ('EZE00100082', ('TMIN', 8.600000000000001)),
 ('ITE00100554', ('TMIN', 23.72))]

## (2) **min_temp**로부터 **온도타입**을 제외하고 **stationID와 섭씨온도**만 포함한 RDD 생성

- `map`을 이용하여, **StationID**를 key로, **섭씨온도**를 value로 변경
- `mapValue`를 이용해 기존 value인 **(온도타입, 섭씨온도)를 섭씨온도**로 변경

In [13]:
station_temp = min_temp.map(lambda x:(x[0],x[1][1]))
# map 함수의 인자는 key+value
# x[0]은 key(StationID), x[1]은 value (온도타입, 섭씨온도)이므로, x[1][1]은 섭씨온도가 됨
station_temp.take(5)

[('ITE00100554', 5.359999999999996),
 ('EZE00100082', 7.699999999999999),
 ('ITE00100554', 9.5),
 ('EZE00100082', 8.600000000000001),
 ('ITE00100554', 23.72)]

In [14]:
station_tmp = min_temp.mapValues(lambda x:x[1])
# mapValues 함수의 인자는 value이므로
# x는 (온도타입, 섭씨온도), x[1]은 섭씨온도를 의미함
station_tmp.take(5)

[('ITE00100554', 5.359999999999996),
 ('EZE00100082', 7.699999999999999),
 ('ITE00100554', 9.5),
 ('EZE00100082', 8.600000000000001),
 ('ITE00100554', 23.72)]

## (3) 각 StationID별 최소 온도 값만 추출
- `reduceByKey`를 이용
- 각 key에 대한 최소값을 찾기 위해서는, 동일한 key (StationID)를 만났을 때, 두 value 중 더 작은 값만 취하면 됨

In [16]:
station_min_temp = station_temp.reduceByKey(lambda val1,val2:min(val1,val2))
# 동일한 key를 갖는 두 개의 value인 val1과 val2중 더 작은 값만을 취함
station_min_temp.take(5)
# 단 두개의 station만 TMIN을 가지고 있음

[('ITE00100554', 5.359999999999996), ('EZE00100082', 7.699999999999999)]

In [17]:
spark.stop()